# Dynamic Selection for Time Series Classification via Meta-Learning

### 1. Dependências

In [ ]:
# Instalar dependências
# %pip install aeon>=0.7.0 scikit-learn>=1.2 numpy>=1.26 pandas>=1.5
# %pip install pywt>=1.5 tslearn>=0.6 scipy>=1.11 tqdm>=4.66

In [ ]:
import numpy as np
import pandas as pd
import pywt
import logging

from scipy.fftpack import fft
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.linear_model import RidgeClassifierCV
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
)
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from tslearn.piecewise import (
    PiecewiseAggregateApproximation,
    SymbolicAggregateApproximation,
)

from aeon.datasets import load_classification
from aeon.datasets.tsc_datasets import univariate_equal_length

from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

### 2. Carregamento de Dados (UCR)

In [ ]:
def load_ucr_dataset(dataset_name: str):
    """Carrega dataset UCR e retorna arrays 2D com labels codificadas."""
    le = LabelEncoder()
    X_train_3d, y_train_raw = load_classification(dataset_name, split="TRAIN")
    X_test_3d, y_test_raw = load_classification(dataset_name, split="test")

    X_train = X_train_3d.reshape(X_train_3d.shape[0], -1)
    X_test = X_test_3d.reshape(X_test_3d.shape[0], -1)

    y_train = le.fit_transform(y_train_raw)
    y_test = le.transform(y_test_raw)

    return X_train, X_test, y_train, y_test

### 3. Representações de Dados

In [ ]:
def select_best_wavelet(X: np.ndarray, candidates=None) -> str:
    """Seleciona wavelet que minimiza variância dos coeficientes de detalhe."""
    if candidates is None:
        candidates = [f"db{i}" for i in range(1, 10)]

    best_wavelet, best_var = None, float("inf")
    for wname in candidates:
        try:
            _, cD = pywt.dwt(X, wname, axis=1, mode="constant")
            var = np.var(cD)
            if var < best_var:
                best_var, best_wavelet = var, wname
        except Exception:
            continue
    return best_wavelet or "db1"

In [ ]:
class RepresentationTransformer:
    """Transforma séries temporais em múltiplos domínios de representação.
    
    Representações:
    - TS   : série z-normalizada
    - FFT  : magnitude da Fast Fourier Transform
    - DWT  : coeficientes de aproximação + detalhe (wavelet)
    - PAA  : Piecewise Aggregate Approximation
    - SAX  : Symbolic Aggregate Approximation
    - DIFF : diferenças de primeira ordem (captura tendências)
    """

    REPR_NAMES = ("TS", "FFT", "DWT", "PAA", "SAX", "DIFF")

    def __init__(self, wavelet: str = "db1"):
        self.wavelet = wavelet
        self._paa = None
        self._sax = None
        self._scaler = TimeSeriesScalerMeanVariance()

    def _safe_segments(self, n_features: int) -> int:
        return max(2, n_features // 4)

    def fit(self, X: np.ndarray):
        n_seg = self._safe_segments(X.shape[1])
        n_sax_alpha = max(2, n_seg)
        self._paa = PiecewiseAggregateApproximation(n_segments=n_seg)
        self._sax = SymbolicAggregateApproximation(
            n_segments=n_seg, alphabet_size_avg=n_sax_alpha
        )
        X_3d = X.reshape(X.shape[0], X.shape[1], 1) if X.ndim == 2 else X
        self._paa.fit(X_3d)
        self._sax.fit(X_3d)
        return self

    def transform(self, X: np.ndarray) -> dict:
        X_3d = X.reshape(X.shape[0], X.shape[1], 1) if X.ndim == 2 else X

        def _scale_2d(arr):
            a3d = arr.reshape(arr.shape[0], arr.shape[1], 1) if arr.ndim == 2 else arr
            scaled = self._scaler.fit_transform(a3d)
            return scaled.reshape(scaled.shape[0], -1)

        # TS
        ts_2d = _scale_2d(X)

        # FFT
        fft_2d = _scale_2d(np.abs(fft(X, axis=1)))

        # DWT
        cA, cD = pywt.dwt(X, self.wavelet, axis=1, mode="constant")
        dwt_2d = _scale_2d(np.hstack((cA, cD)))

        # PAA
        paa_inv = self._paa.inverse_transform(self._paa.transform(X_3d))
        paa_2d = paa_inv.reshape(paa_inv.shape[0], -1)

        # SAX
        sax_inv = self._sax.inverse_transform(self._sax.transform(X_3d))
        sax_2d = sax_inv.reshape(sax_inv.shape[0], -1)

        # DIFF
        diff_2d = _scale_2d(np.diff(X, axis=1))

        return {
            "TS": ts_2d, "FFT": fft_2d, "DWT": dwt_2d,
            "PAA": paa_2d, "SAX": sax_2d, "DIFF": diff_2d,
        }

    def fit_transform(self, X: np.ndarray) -> dict:
        return self.fit(X).transform(X)

### 4. Pool de Classificadores Base

In [ ]:
def build_classifier_pool() -> dict:
    return {
        "RF": RandomForestClassifier(
            n_estimators=300, max_features="sqrt", n_jobs=-1, random_state=42
        ),
        "ET": ExtraTreesClassifier(
            n_estimators=300, max_features="sqrt", n_jobs=-1, random_state=42
        ),
        "GBM": GradientBoostingClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42
        ),
        "SVM": SVC(probability=True, random_state=42),
    }


def build_svm_param_grid() -> dict:
    return {
        "kernel": ["rbf", "linear", "poly"],
        "C": [0.1, 1, 10, 100],
        "gamma": ["scale", "auto"],
    }

### 5. CAWPE – Cross-validation Accuracy Weighted Probabilistic Ensemble

In [ ]:
class CAWPE:
    """Cross-validation Accuracy Weighted Probabilistic Ensemble.
    
    Pesos são estimados por validação cruzada e elevados a alpha
    para amplificar diferenças entre classificadores melhores e piores.
    """

    def __init__(self, n_folds=5, alpha=4.0, tune_svm=True, random_state=42):
        self.n_folds = n_folds
        self.alpha = alpha
        self.tune_svm = tune_svm
        self.random_state = random_state
        self._repr_classifiers = {}
        self._weights = {}
        self._n_classes = 0

    def fit(self, repr_data: dict, y: np.ndarray):
        self._n_classes = len(np.unique(y))
        skf = StratifiedKFold(
            n_splits=self.n_folds, shuffle=True, random_state=self.random_state
        )

        for repr_name, X_repr in repr_data.items():
            self._repr_classifiers[repr_name] = {}
            self._weights[repr_name] = {}

            pool = build_classifier_pool()
            for clf_name, clf in pool.items():
                # Tuning SVM
                if clf_name == "SVM" and self.tune_svm:
                    clf = GridSearchCV(
                        clf, build_svm_param_grid(),
                        cv=3, scoring="accuracy", n_jobs=-1, refit=True,
                    )

                # Acurácia por validação cruzada
                fold_accs = []
                for tr_idx, val_idx in skf.split(X_repr, y):
                    c = clone(clf)
                    c.fit(X_repr[tr_idx], y[tr_idx])
                    fold_accs.append(accuracy_score(y[val_idx], c.predict(X_repr[val_idx])))

                cv_acc = np.mean(fold_accs)
                self._weights[repr_name][clf_name] = cv_acc ** self.alpha

                # Retreinar no dataset completo
                final_clf = clone(clf)
                final_clf.fit(X_repr, y)
                self._repr_classifiers[repr_name][clf_name] = final_clf

        # Normalizar pesos globalmente
        total = sum(w for rw in self._weights.values() for w in rw.values())
        if total > 0:
            for rn in self._weights:
                for cn in self._weights[rn]:
                    self._weights[rn][cn] /= total
        return self

    def predict_proba(self, repr_data: dict) -> np.ndarray:
        n = next(iter(repr_data.values())).shape[0]
        combined = np.zeros((n, self._n_classes))
        for rn, X_r in repr_data.items():
            if rn not in self._repr_classifiers:
                continue
            for cn, clf in self._repr_classifiers[rn].items():
                w = self._weights[rn][cn]
                p = clf.predict_proba(X_r)
                if p.shape[1] < self._n_classes:
                    fp = np.zeros((n, self._n_classes))
                    fp[:, :p.shape[1]] = p
                    p = fp
                combined += w * p
        sums = combined.sum(axis=1, keepdims=True)
        sums[sums == 0] = 1.0
        return combined / sums

    def predict(self, repr_data: dict) -> np.ndarray:
        return np.argmax(self.predict_proba(repr_data), axis=1)

    def get_per_repr_proba(self, repr_data: dict) -> dict:
        """Retorna probabilidades agregadas POR REPRESENTAÇÃO.
        
        Essencial para o meta-classifier: ele recebe as probabilities
        separadas por representação e aprende qual domínio confiar
        para cada tipo de instância.
        """
        n = next(iter(repr_data.values())).shape[0]
        result = {}
        for rn, X_r in repr_data.items():
            if rn not in self._repr_classifiers:
                continue
            rp = np.zeros((n, self._n_classes))
            rw = 0.0
            for cn, clf in self._repr_classifiers[rn].items():
                w = self._weights[rn][cn]
                p = clf.predict_proba(X_r)
                if p.shape[1] < self._n_classes:
                    fp = np.zeros((n, self._n_classes))
                    fp[:, :p.shape[1]] = p
                    p = fp
                rp += w * p
                rw += w
            if rw > 0:
                rp /= rw
            result[rn] = rp
        return result

### 6. Meta-Learner com Stacking Cross-Validado

In [ ]:
class DynamicSelectionMetaLearner(BaseEstimator, ClassifierMixin):
    """Modelo de seleção dinâmica com dois níveis de stacking.
    
    Nível 0: CAWPE multi-representação
    Nível 1: Meta-classifier treinado em meta-features out-of-fold
    
    As meta-features são probabilidades por representação concatenadas,
    permitindo que o meta-classifier aprenda QUAL representação é mais
    confiável para cada tipo de instância.
    """

    def __init__(self, n_folds_cawpe=5, n_folds_meta=5, alpha=4.0,
                 tune_svm=True, random_state=42):
        self.n_folds_cawpe = n_folds_cawpe
        self.n_folds_meta = n_folds_meta
        self.alpha = alpha
        self.tune_svm = tune_svm
        self.random_state = random_state
        self._repr_transformer = None
        self._cawpe = None
        self._meta_clf = None
        self._scaler = MinMaxScaler()
        self._n_classes = 0

    def fit(self, X, y):
        self._n_classes = len(np.unique(y))

        # 1. Wavelet
        logger.info("Step 1/5: Selecionando melhor wavelet...")
        best_wl = select_best_wavelet(X)
        logger.info(f"  Wavelet: {best_wl}")

        # 2. Representações
        logger.info("Step 2/5: Gerando representações...")
        self._repr_transformer = RepresentationTransformer(wavelet=best_wl)
        repr_data = self._repr_transformer.fit_transform(X)

        # 3. Meta-features out-of-fold
        logger.info("Step 3/5: Gerando meta-features out-of-fold...")
        oof_meta = self._generate_oof_meta_features(repr_data, y)

        # 4. Meta-classifier
        logger.info("Step 4/5: Treinando meta-classifier...")
        oof_scaled = self._scaler.fit_transform(oof_meta)
        self._meta_clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
        self._meta_clf.fit(oof_scaled, y)

        # 5. CAWPE final
        logger.info("Step 5/5: Retreinando CAWPE no dataset completo...")
        self._cawpe = CAWPE(
            n_folds=self.n_folds_cawpe, alpha=self.alpha,
            tune_svm=self.tune_svm, random_state=self.random_state,
        )
        self._cawpe.fit(repr_data, y)
        logger.info("Treino completo.")
        return self

    def predict(self, X):
        repr_data = self._repr_transformer.transform(X)
        meta_feat = self._build_meta_features(repr_data)
        return self._meta_clf.predict(self._scaler.transform(meta_feat))

    def predict_ensemble(self, X):
        """Predição direta via CAWPE (sem meta-level)."""
        repr_data = self._repr_transformer.transform(X)
        return self._cawpe.predict(repr_data)

    def _generate_oof_meta_features(self, repr_data, y):
        n = y.shape[0]
        repr_names = list(repr_data.keys())
        n_cols = len(repr_names) * self._n_classes
        oof = np.zeros((n, n_cols))

        skf = StratifiedKFold(
            n_splits=self.n_folds_meta, shuffle=True, random_state=self.random_state
        )

        for fi, (tr_idx, val_idx) in enumerate(skf.split(oof, y)):
            logger.info(f"  Meta-fold {fi+1}/{self.n_folds_meta}")
            repr_tr = {k: v[tr_idx] for k, v in repr_data.items()}
            repr_val = {k: v[val_idx] for k, v in repr_data.items()}

            n_folds_inner = min(self.n_folds_cawpe, len(np.unique(y[tr_idx])))
            fold_cawpe = CAWPE(
                n_folds=n_folds_inner, alpha=self.alpha,
                tune_svm=self.tune_svm, random_state=self.random_state + fi,
            )
            fold_cawpe.fit(repr_tr, y[tr_idx])

            per_repr = fold_cawpe.get_per_repr_proba(repr_val)
            row = np.hstack([per_repr[rn] for rn in repr_names if rn in per_repr])
            oof[val_idx] = row

        return oof

    def _build_meta_features(self, repr_data):
        per_repr = self._cawpe.get_per_repr_proba(repr_data)
        repr_names = list(repr_data.keys())
        return np.hstack([per_repr[rn] for rn in repr_names if rn in per_repr])

### 7. Modelo Ensemble Direto (CAWPE sem meta-level)

Variante simplificada

In [ ]:
class DynamicSelectionEnsemble(BaseEstimator, ClassifierMixin):
    """CAWPE multi-representação sem meta-classifier."""

    def __init__(self, n_folds=5, alpha=4.0, tune_svm=True, random_state=42):
        self.n_folds = n_folds
        self.alpha = alpha
        self.tune_svm = tune_svm
        self.random_state = random_state
        self._repr_transformer = None
        self._cawpe = None

    def fit(self, X, y):
        best_wl = select_best_wavelet(X)
        self._repr_transformer = RepresentationTransformer(wavelet=best_wl)
        repr_data = self._repr_transformer.fit_transform(X)
        self._cawpe = CAWPE(
            n_folds=self.n_folds, alpha=self.alpha,
            tune_svm=self.tune_svm, random_state=self.random_state,
        )
        self._cawpe.fit(repr_data, y)
        return self

    def predict(self, X):
        return self._cawpe.predict(self._repr_transformer.transform(X))

    def predict_proba(self, X):
        return self._cawpe.predict_proba(self._repr_transformer.transform(X))

### 8. Execução dos Experimentos

In [ ]:
def run_experiment(dataset_names=None, model_type="meta", alpha=4.0,
                   tune_svm=True, n_folds_cawpe=5, n_folds_meta=5):
    if dataset_names is None:
        dataset_names = list(univariate_equal_length)

    results = []
    for ds in tqdm(dataset_names, desc="Datasets"):
        try:
            X_train, X_test, y_train, y_test = load_ucr_dataset(ds)

            if model_type == "meta":
                model = DynamicSelectionMetaLearner(
                    n_folds_cawpe=n_folds_cawpe, n_folds_meta=n_folds_meta,
                    alpha=alpha, tune_svm=tune_svm,
                )
            else:
                model = DynamicSelectionEnsemble(
                    n_folds=n_folds_cawpe, alpha=alpha, tune_svm=tune_svm,
                )

            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            acc = accuracy_score(y_test, preds)
            results.append({"Dataset": ds, "Accuracy": acc})
            print(f"  {ds}: {acc:.4f}")
        except Exception as e:
            print(f"  FALHOU em {ds}: {e}")
            results.append({"Dataset": ds, "Accuracy": np.nan})

    df = pd.DataFrame(results)
    print(f"\nAcurácia média: {df['Accuracy'].mean():.4f} ± {df['Accuracy'].std():.4f}")
    return df

### 9. Teste Rápido

In [ ]:
QUICK_TEST = [
    "ArrowHead", "Car",
    "ChlorineConcentration", "ECG200",
]

print("=" * 60)
print("Meta-Learner Model")
print("=" * 60)
df_meta = run_experiment(QUICK_TEST, model_type="meta")
display(df_meta)

In [ ]:
print("=" * 60)
print("Ensemble Model (CAWPE sem meta-classifier)")
print("=" * 60)
df_ensemble = run_experiment(QUICK_TEST, model_type="ensemble")
display(df_ensemble)

### 10. Execução Completa (112 datasets UCR)

In [ ]:
# ATENÇÃO: Isso pode levar muitas horas para executar
# df_full_meta = run_experiment(model_type="meta")
# df_full_meta.to_csv("results_meta_full.csv", index=False)

# df_full_ensemble = run_experiment(model_type="ensemble")
# df_full_ensemble.to_csv("results_ensemble_full.csv", index=False)